transformers
<br>
Hugging Face에서 개발한 메인 AI 라이브러리입니다. <br>최신 LLM(Llama, Mistral 등), BERT, Vision Transformer 등 수만 개의 사전 학습된 AI 모델과 pipeline() 도구를 파이썬 코드로 불러올 수 있게 해줍니다.
<br>
<br>
torch (PyTorch)
<br>
Meta(Facebook)에서 개발한 오픈소스 딥러닝 연산 프레임워크입니다. <br>AI 모델 내부의 복잡한 행렬 연산과 가중치 계산, GPU 가속을 실제로 처리하는 백엔드 엔진 역할을 합니다.

In [ ]:
!pip install transformers torch

가장 간단하게 사전 학습된 모델을 불러와 감정 분석(Sentiment Analysis)

In [1]:
from transformers import pipeline

# 1. 감정 분석 전용 파이프라인 생성 (기본 모델 자동 다운로드)
classifier = pipeline("sentiment-analysis")

# 2. 분석할 입력 데이터 작성
text = "나는 파이썬 공부하는것이 좋아요!"

# 3. 파이프라인 실행 및 결과 출력
result = classifier(text)

print(f"입력문: {text}")
print(f"분석 결과: {result}")

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

입력문: 나는 파이썬 공부하는것이 좋아요!
분석 결과: [{'label': 'POSITIVE', 'score': 0.9699311852455139}]


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"


# Step 1: 전처리 (Tokenizer)

tokenizer = AutoTokenizer.from_pretrained(model_name)

raw_text = "This workbook makes learning AI so easy!"
# 텍스트를 파이토치 텐서 형태로 변환
inputs = tokenizer(raw_text, return_tensors="pt")

print("--- [1단계: Tokenizer 결과] ---")
print("Input IDs (숫자로 변환된 토큰):", inputs["input_ids"])
print("Attention Mask:", inputs["attention_mask"])



# Step 2: 모델 추론 (Model)

model = AutoModelForSequenceClassification.from_pretrained(model_name)

with torch.no_grad():
    outputs = model(**inputs)

print("\n--- [2단계: Model 출력 결과] ---")
print("Raw Logits (후처리 전 출력값):", outputs.logits)



# Step 3: 후처리 (Post-processing)

# 1. Logits에 Softmax를 적용하여 확률(Probability)값으로 변환
predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)

# 2. 가장 높은 확률을 가진 인덱스 추출
predicted_class_id = torch.argmax(predictions, dim=-1).item()

# 3. 인덱스를 클래스 라벨(POSITIVE/NEGATIVE)로 변환
label = model.config.id2label[predicted_class_id]
score = predictions[0][predicted_class_id].item()

print("\n--- [3단계: 최종 분석 결과] ---")
print(f"최종 예측 라벨: {label}")
print(f"확신도 (Score): {score:.4f}")

--- [1단계: Tokenizer 결과] ---
Input IDs (숫자로 변환된 토큰): tensor([[ 101, 2023, 2147, 8654, 3084, 4083, 9932, 2061, 3733,  999,  102]])
Attention Mask: tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


--- [2단계: Model 출력 결과] ---
Raw Logits (후처리 전 출력값): tensor([[-2.7625,  2.7368]])

--- [3단계: 최종 분석 결과] ---
최종 예측 라벨: POSITIVE
확신도 (Score): 0.9959


**유해/공격성 텍스트 분류 모델** unitary/toxic-bert
<br>
<br>
온라인 댓글 및 토론 데이터를 바탕으로 학습된 멀티 레이블(Multi-label) 텍스트 분류 모델입니다. 단순 욕설뿐만 아니라 위협(threat), 모욕(insult), 정체성 증오(identity hate) 등을 종합 검출합니다.  
<br>
AI 입출력 모더레이션, 커뮤니티 악성 유저 및 폭력성 메시지 자동 필터링.

In [ ]:
from transformers import pipeline

# ToxicBERT 분류 파이프라인 (top_k=None으로 설정하여 모든 항목 점수 반환)
toxic_classifier = pipeline(
    "text-classification",
    model="unitary/toxic-bert",
    top_k=None
)

comment = "I will hack your account and delete all your files!"
results = toxic_classifier(comment)[0]

print(f"[입력문]: {comment}")
print("[유해성 상세 진단]")
for item in results:
    label = item['label']
    score = item['score']
    if score > 0.5:  # 50% 이상 확률로 판정된 유해 레이블 출력
        print(f" - {label}: {score * 100:.2f}%")

config.json:   0%|          | 0.00/811 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/174 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

[입력문]: I will hack your account and delete all your files!
[유해성 상세 진단]
 - toxic: 82.27%


**피싱 URL 탐지 모델: darshan8950/phishing_url_detection_BERT**
<br>
<br>
BERT 기반 모델을 URL 피싱 classification 데이터셋으로 파인튜닝한 모델입니다. 웹 주소(URL)의 구조와 패턴을 분석하여 해당 링크가 피싱/악성 사이트인지 판별합니다.  
<br>

이메일 및 메시징 보안 솔루션, 악성 URL 자동 차단 필터.

In [ ]:
from transformers import pipeline

# 피싱 URL 분류 파이프라인
phishing_detector = pipeline(
    "text-classification",
    model="darshan8950/phishing_url_detection_BERT"
)

urls = [
    "https://www.google.com/search?q=huggingface",
    "http://login-verify-account-security-alert.free-domain.com/auth/login",
    "https://www.naver.com",
    "https://github.com/search?q=cnn",
    "http://paypal.com.account-verification-service.top/login.php",
    "http://192.168.1.50/admin/download/payload.exe"
]

for url in urls:
    prediction = phishing_detector(url)[0]
    label = prediction['label']
    score = prediction['score']

    status = "⚠️ 피싱/위험" if label in ["LABEL_1", "1", "PHISHING"] else "✅ 안전"
    print(f"[URL]: {url}")
    print(f"[검사 결과]: {status} (확신도: {score:.4f})\n")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[URL]: https://www.google.com/search?q=huggingface
[검사 결과]: ✅ 안전 (확신도: 0.9961)

[URL]: http://login-verify-account-security-alert.free-domain.com/auth/login
[검사 결과]: ✅ 안전 (확신도: 1.0000)

[URL]: https://www.naver.com
[검사 결과]: ✅ 안전 (확신도: 1.0000)

[URL]: https://github.com/search?q=cnn
[검사 결과]: ✅ 안전 (확신도: 0.9959)

[URL]: http://paypal.com.account-verification-service.top/login.php
[검사 결과]: ✅ 안전 (확신도: 1.0000)

[URL]: http://192.168.1.50/admin/download/payload.exe
[검사 결과]: ✅ 안전 (확신도: 1.0000)



**소스코드 취약점 탐지 모델: mrm8488/codebert-base-finetuned-detect-insecure-code**
<br>
<br>
Microsoft의 CodeBERT를 기반으로 파인튜닝된 모델로, 소스코드(C, C++, Java, Python 등)에 버퍼 오버플로우, 메모리 누수, DoS 위험 등의 보안 취약점이 포함되어 있는지 분류합니다.
<br>
CI/CD 파이프라인 내 정적 코드 분석(SAST), DevSecOps 자동 검사.


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "mrm8488/codebert-base-finetuned-detect-insecure-code"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# 진단할 코드 조각 예시 (취약한 C 코드)
insecure_code = """
void process_user_input(char *user_input) {
    char buffer[16];
    strcpy(buffer, user_input); // 경계 검사가 없어 버퍼 오버플로우 가능
}
"""

inputs = tokenizer(insecure_code, return_tensors="pt", truncation=True, padding=True)

with torch.no_grad():
    logits = model(**inputs).logits
    prediction = torch.argmax(logits, dim=-1).item()

# 0: Secure(안전), 1: Insecure(취약)
result_str = "취약점 감지됨 (Insecure)" if prediction == 1 else "안전한 코드 (Secure)"
print(f"코드 검사 진단: {result_str}")

config.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.22k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

...

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

코드 검사 진단: 취약점 감지됨 (Insecure)


실시간 입력값 SQL 인젝션 탐지: cssupport/mobilebert-sql-injection-detect
<br>
<br>
MobileBERT 기반으로 학습된 약 100MB 규모의 초경량 모델입니다. <br>웹 폼 입력값이나 URL 파라미터에 ' OR '1'='1 같은 SQL 인젝션 공격 페이로드가 포함되어 있는지 빠르게 판별합니다.
<br>
<br>WAF(웹 애플리케이션 방화벽) 보조 필터, 웹 서비스 로그인/검색 폼 실시간 차단.  


In [ ]:
from transformers import pipeline

# MobileBERT 기반 SQL 인젝션 탐지 파이프라인 로드
sqli_detector = pipeline("text-classification", model="cssupport/mobilebert-sql-injection-detect")

# 검사할 사용자 입력 예시
test_inputs = [
    "user_email@example.com",                        # 정상 입력
    "admin' OR '1'='1' --",                          # 인증 우회 SQLi
    "1; DROP TABLE users;"                           # 악의적 쿼리 삽입
]

for text in test_inputs:
    result = sqli_detector(text)[0]
    label = result['label']
    score = result['score']

    status = "⚠️ SQL 인젝션 감지" if label == "LABEL_1" else "✅ 정상 입력"
    print(f"[입력문]: {text}")
    print(f"[검사 결과]: {status} (확신도: {score:.4f})\n")

config.json:   0%|          | 0.00/2.09k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 98.8MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/1113 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 98.5MB            

model.safetensors: downloading bytes:           |  0.00B            

tokenizer_config.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

[입력문]: user_email@example.com
[검사 결과]: ✅ 정상 입력 (확신도: 0.9731)

[입력문]: admin' OR '1'='1' --
[검사 결과]: ⚠️ SQL 인젝션 감지 (확신도: 1.0000)

[입력문]: 1; DROP TABLE users;
[검사 결과]: ✅ 정상 입력 (확신도: 0.9972)



**SQL 쿼리 안전성/취약점 분류: salmane11/SQLQueryShield**
<br>
<br>
CodeBERT를 기반으로 파인튜닝된 모델로, 생성되거나 실행되려는 SQL 쿼리 자체를 분석하여 유해 여부(UNION 공격, 무단 데이터 탈취, 구조 파괴 등)를 진단합니다.  <br>
<br>
Text-to-SQL AI 서비스의 출력 쿼리 검증, DB 실행 전 보안 가드레일.  

In [ ]:
from transformers import pipeline

# CodeBERT 기반 SQL 쿼리 검증 파이프라인 로드
sql_shield = pipeline("text-classification", model="salmane11/SQLQueryShield")

queries = [
    "SELECT name, department FROM employees WHERE id = 101;",  # 정상 쿼리
    "SELECT name FROM employees WHERE id = '' UNION SELECT database() --" # 데이터 탈취형 공격 쿼리
]

for q in queries:
    prediction = sql_shield(q)[0]
    label = prediction['label']  # 'BENIGN' 또는 'MALICIOUS'
    score = prediction['score']

    print(f"[SQL 쿼리]: {q}")
    print(f"[진단 결과]: {label} (확신도: {score:.4f})\n")

config.json:   0%|          | 0.00/883 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.22k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/958 [00:00<?, ?B/s]

[SQL 쿼리]: SELECT name, department FROM employees WHERE id = 101;
[진단 결과]: SAFE (확신도: 0.5242)

[SQL 쿼리]: SELECT name FROM employees WHERE id = '' UNION SELECT database() --
[진단 결과]: MALICIOUS (확신도: 0.9995)



**초경량 스미싱/스팸 탐지: mrm8488/bert-tiny-finetuned-sms-spam-detection**
<br>
<br>
파라미터 수가 440만 개에 불과한 BERT-Tiny 기반의 초경량 모델입니다. 문자 메시지(SMS) 내 스미싱, 피싱 링크, 악성 광고 스팸 여부를 매우 빠른 속도로 분류합니다.  <br>
<br>
모바일 앱 필터, 이메일/SMS 스미싱 차단 미들웨어.  

In [ ]:
from transformers import pipeline

# SMS 스팸/스미싱 분류 파이프라인
sms_detector = pipeline(
    "text-classification",
    model="mrm8488/bert-tiny-finetuned-sms-spam-detection"
)

messages = [
    "Hey, are we still meeting for lunch today?", # 정상 메시지
    "URGENT! You have won $1,000 gift card. Click http://bit.ly/claim-now to get it!" # 스미싱
]

for msg in messages:
    result = sms_detector(msg)[0]
    label = result['label']  # 'LABEL_1': 스팸/스미싱, 'LABEL_0': 정상
    score = result['score']

    status = "⚠️ 스미싱/스팸 위험" if label == "LABEL_1" else "✅ 정상 메시지"
    print(f"[메시지]: {msg}")
    print(f"[검사 결과]: {status} (확신도: {score:.4f})\n")

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 17.6MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/41 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/324 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

[메시지]: Hey, are we still meeting for lunch today?
[검사 결과]: ✅ 정상 메시지 (확신도: 0.9368)

[메시지]: URGENT! You have won $1,000 gift card. Click http://bit.ly/claim-now to get it!
[검사 결과]: ✅ 정상 메시지 (확신도: 0.8216)

